In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from AttackInfo import Sensors, Actuators

In [3]:
df_normal = pd.read_parquet('../../Dataset/SWaT_Dataset_Normal_v1.parquet')
df_attack  = pd.read_parquet('../../Dataset/SWaT_Dataset_Attack_v1.parquet')

In [4]:
# 3.1 缺失值檢查
print("=== 缺失值檢查 ===")
print(f"Normal  NaN 總計: {df_normal.isnull().sum().sum()}")
print(f"Attack  NaN 總計: {df_attack.isnull().sum().sum()}")

=== 缺失值檢查 ===
Normal  NaN 總計: 0
Attack  NaN 總計: 0


In [5]:
# 3.2 Timestamp 連續性檢查
def check_continuity(df, name):
    ts = df['Timestamp'].sort_values().reset_index(drop=True)
    diff = ts.diff().iloc[1:]
    gaps = diff[diff != pd.Timedelta(seconds=1)]
    if gaps.empty:
        print(f"✅ {name}: 時間完全連續")
    else:
        print(f"❌ {name}: 發現 {len(gaps)} 處斷點")
        for idx in gaps.index[:5]:
            print(f"   {ts[idx-1]}  →  {ts[idx]}  (間隔 {gaps[idx].total_seconds():.0f} 秒)")

check_continuity(df_normal, 'Normal_v1')
check_continuity(df_attack,  'Attack_v1')

✅ Normal_v1: 時間完全連續
❌ Attack_v1: 發現 1 處斷點
   2015-12-31 21:00:36  →  2015-12-31 21:01:58  (間隔 82 秒)


In [6]:
# 3.3 感測器數值範圍（Normal 資料）
print("=== 感測器數值統計（Normal_v1）===")
sensor_stats = df_normal[Sensors].describe().T
sensor_stats['range'] = sensor_stats['max'] - sensor_stats['min']
display(sensor_stats[['mean', 'std', 'min', 'max', 'range']].round(4))

=== 感測器數值統計（Normal_v1）===


,mean,std,min,max,range
FIT101,1.8505,1.1325,0.0000,2.7451,2.7451
LIT101,587.5328,121.6665,120.6237,817.5565,696.9328
AIT201,263.7835,4.7871,251.6662,272.5263,20.8601
AIT202,8.3882,0.0902,8.2587,8.9883,0.7296
AIT203,348.3793,49.4500,312.2789,567.4699,255.1910
FIT201,1.8341,1.0593,0.0000,2.4879,2.4879
DPIT301,16.6047,6.7300,0.0000,21.0993,21.0993
FIT301,1.8373,0.8209,0.0000,2.3588,2.3588
LIT301,899.8929,93.9128,132.8185,1014.7240,881.9055
AIT401,124.7171,54.8083,0.0000,148.8561,148.8561


In [10]:
# 3.4 致動器狀態檢查（三態確認）
print("=== 致動器唯一值 ===")
act_summary = []
for col in Actuators:
    if col in df_attack.columns:
        vals_n = sorted(df_normal[col].unique())
        vals_a = sorted(df_attack[col].unique())
        act_summary.append({'Actuator': col,
                            'Normal unique': vals_n,
                            'Attack unique': vals_a,
                            'New in Attack?': [v for v in vals_a if v not in vals_n]})
summary_df = pd.DataFrame(act_summary)
# 只顯示在攻擊資料中出現新狀態的致動器
new_vals = summary_df[summary_df['New in Attack?'].apply(len) > 0]
if new_vals.empty:
    print("✅ 所有致動器的狀態值在攻擊資料中無新增（與 Normal 一致）")
else:
    print("⚠️ 以下致動器在攻擊資料中出現了 Normal 中沒有的狀態：")
    display(new_vals)
print()
print("MV 系列（電動閥）：0=關閉, 1=開啟, 2=切換中")
print("P  系列（泵浦）  ：1=關閉, 2=運作")
print("UV 系列（紫外線）：1=關閉, 2=運作")

# 顯示每個致動器的完整狀態清單
print()
for col in Actuators:
    if col in df_attack.columns:
        vals = sorted(df_attack[col].unique())
        print(f"  {col}: {vals}")

=== 致動器唯一值 ===
⚠️ 以下致動器在攻擊資料中出現了 Normal 中沒有的狀態：


,Actuator,Normal unique,Attack unique,New in Attack?
2,P102,[1],"[1, 2]",[2]
4,P201,[1],"[1, 2]",[2]
7,P204,[1],"[1, 2]",[2]
9,P206,[1],"[1, 2]",[2]
18,P403,[1],"[1, 2]",[2]



MV 系列（電動閥）：0=關閉, 1=開啟, 2=切換中
P  系列（泵浦）  ：1=關閉, 2=運作
UV 系列（紫外線）：1=關閉, 2=運作

  MV101: [np.int64(0), np.int64(1), np.int64(2)]
  P101: [np.int64(1), np.int64(2)]
  P102: [np.int64(1), np.int64(2)]
  MV201: [np.int64(0), np.int64(1), np.int64(2)]
  P201: [np.int64(1), np.int64(2)]
  P202: [np.int64(1)]
  P203: [np.int64(1), np.int64(2)]
  P204: [np.int64(1), np.int64(2)]
  P205: [np.int64(1), np.int64(2)]
  P206: [np.int64(1), np.int64(2)]
  MV301: [np.int64(0), np.int64(1), np.int64(2)]
  MV302: [np.int64(0), np.int64(1), np.int64(2)]
  MV303: [np.int64(0), np.int64(1), np.int64(2)]
  MV304: [np.int64(0), np.int64(1), np.int64(2)]
  P301: [np.int64(1)]
  P302: [np.int64(1), np.int64(2)]
  P401: [np.int64(1)]
  P402: [np.int64(1), np.int64(2)]
  P403: [np.int64(1), np.int64(2)]
  P404: [np.int64(1)]
  UV401: [np.int64(1), np.int64(2)]
  P501: [np.int64(1), np.int64(2)]
  P502: [np.int64(1)]
  P601: [np.int64(1)]
  P602: [np.int64(1), np.int64(2)]
  P603: [np.int64(1)]
